In [1]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv("customer_shopping_behavior.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
df.shape

(3900, 18)

In [11]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [29]:
df["Review Rating"]=df.groupby("Category")["Review Rating"].transform(lambda x : x.fillna(x.median()))

In [39]:
df.columns= df.columns.str.lower()
df.columns= df.columns.str.replace(" ","_")

In [31]:
df = df.rename(columns={ 'purchase_amount_(usd)': 'purchase_amount'})

In [32]:
#creating a new column
labels = ["Young Adult","Adult","Middle-aged","Senior"]
df["age_group"]=pd.qcut(df["age"],q=4,labels = labels)

In [20]:
df[["age","age_group"]].sample(10)

,age,age_group
2422,30,Young Adult
1946,59,Senior
448,32,Adult
3858,30,Young Adult
814,31,Young Adult
2377,44,Adult
3781,52,Middle-aged
3722,23,Young Adult
803,32,Adult
1822,66,Senior


In [33]:
# create column purchase_frequency_days
frequency_mapping ={
    "Fortnightly":14,
    "Weekly":7,
    "Monthly":30,
    "Quarterly":90,
    "Bi-Weekly":14,
    "Annually":365,
    "Every 3 Months":90
}
df["purchase_frequency_days"]=df["frequency_of_purchases"].map(frequency_mapping)

In [41]:
df[["purchase_frequency_days","frequency_of_purchases"]].sample(10)

,purchase_frequency_days,frequency_of_purchases
1916,14,Bi-Weekly
3223,14,Bi-Weekly
3193,14,Fortnightly
3716,90,Every 3 Months
2190,365,Annually
2302,30,Monthly
58,30,Monthly
2894,365,Annually
2290,90,Quarterly
3081,7,Weekly


In [ ]:
#checkig for redundant column
(df["discount_applied"]==df["promo_code_used"]).all()

np.True_

In [34]:
df=df.drop("promo_code_used",axis=1)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [47]:
%pip install pandas sqlalchemy pyodbc

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 5.6 MB/s eta 0:00:01
   ---------------------------------- ----- 2.1/2.4 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.8 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ----------------------------

In [35]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from urllib.parse import quote_plus

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=LAPTOP-D98GQN8T\\MUQTADIRSERVER;"
    "DATABASE=customer_behaviour;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)

engine = create_engine(connection_url)

In [36]:
with engine.connect() as conn:
    print(conn.exec_driver_sql("SELECT DB_NAME()").scalar())

customer_behaviour


In [37]:
df.to_sql(
    name="customer_shopping_behavior",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False
)

50

In [24]:
pd.read_sql_query(
    "SELECT TOP 10 * FROM dbo.customer_shopping_behavior",
    engine
)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,Yes,14,Venmo,Weekly
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,Yes,49,Cash,Quarterly
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,Yes,19,Credit Card,Weekly
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,Yes,8,Venmo,Annually
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,Yes,4,Cash,Quarterly


In [38]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM df",
    engine
)

,total_rows
0,3900


In [26]:
query = """
SELECT
    Gender,
    COUNT(*) AS customer_count
FROM df
GROUP BY Gender;
"""

result = pd.read_sql_query(query, engine)
result

,Gender,customer_count
0,Male,2652
1,Female,1248


In [27]:
query = """
SELECT
    *
FROM customer_shopping_behavior;
"""

result = pd.read_sql_query(query, engine)
result

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [28]:
query="""
select top 5 * from customer_shopping_behavior
"""
result=pd.read_sql_query(query,engine)
result

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
